In [2]:
import os
print(os.getcwd())

d:\Masai\project 2\Capstone_Part2


In [3]:
import os
print(os.listdir())

['rfm_segmentation.ipynb']


In [4]:
import os

print("Current Folder:", os.getcwd())
print("\nFiles & Folders:")

for item in os.listdir():
    print(item)

Current Folder: d:\Masai\project 2\Capstone_Part2

Files & Folders:
rfm_segmentation.ipynb


In [5]:
import os

print(os.path.abspath("."))
print(os.path.exists("source"))

d:\Masai\project 2\Capstone_Part2
False


In [7]:
import os

for root, dirs, files in os.walk(r"D:\Masai"):
    if "customers.csv" in files:
        print(os.path.join(root, "customers.csv"))

D:\Masai\project 2\source\customers.csv


In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

customers = pd.read_csv("../source/customers.csv")
orders = pd.read_csv("../source/orders.csv")
tickets = pd.read_csv("../source/support_tickets.csv")
web = pd.read_csv("../source/web_events_snapshot.csv")
campaign = pd.read_csv("../source/intervention_history.csv")

print(customers.shape)
print(orders.shape)

(2400, 9)
(10009, 10)


In [9]:
print("CUSTOMERS")
print(customers.columns.tolist())

print("\nORDERS")
print(orders.columns.tolist())

print("\nTICKETS")
print(tickets.columns.tolist())

print("\nWEB")
print(web.columns.tolist())

print("\nCAMPAIGN")
print(campaign.columns.tolist())

CUSTOMERS
['customer_id', 'signup_date', 'city_tier', 'age_group', 'acquisition_channel', 'loyalty_tier', 'preferred_category', 'skin_type', 'marketing_consent']

ORDERS
['order_id', 'customer_id', 'order_date', 'category', 'quantity', 'gross_amount', 'discount_pct', 'delivery_days', 'returned', 'rating']

TICKETS
['ticket_id', 'customer_id', 'ticket_date', 'issue_type', 'support_channel', 'resolution_hours', 'sentiment_score', 'reopened']

WEB
['customer_id', 'snapshot_date', 'sessions_30d', 'product_views_30d', 'cart_adds_30d', 'wishlist_adds_30d', 'abandoned_carts_30d', 'email_opens_30d', 'campaign_clicks_30d', 'last_visit_days_ago']

CAMPAIGN
['customer_id', 'snapshot_date', 'last_campaign_received', 'last_campaign_cost', 'manual_priority_bucket']


In [10]:
# Convert date columns
orders['order_date'] = pd.to_datetime(orders['order_date'])

# Snapshot date (latest order date)
snapshot_date = orders['order_date'].max() + pd.Timedelta(days=1)

# RFM calculation
rfm = orders.groupby('customer_id').agg({
    'order_date': lambda x: (snapshot_date - x.max()).days,
    'order_id': 'count',
    'gross_amount': 'sum'
}).reset_index()

rfm.columns = ['customer_id', 'recency', 'frequency', 'monetary']

rfm.head()

,customer_id,recency,frequency,monetary
0,CUST00001,168,6,2955.57
1,CUST00002,35,3,1713.10
2,CUST00003,232,1,649.98
3,CUST00004,192,1,1604.04
4,CUST00005,11,6,3910.43


In [11]:
print(rfm.shape)
print(rfm[['customer_id','recency','frequency','monetary']].describe())

(2400, 4)
           recency    frequency      monetary
count  2400.000000  2400.000000   2400.000000
mean    106.552083     4.170417   3102.366721
std     103.035447     2.626960   2318.314239
min       1.000000     1.000000    149.000000
25%      23.000000     2.000000   1371.697500
50%      56.000000     4.000000   2621.735000
75%     180.250000     6.000000   4253.452500
max     623.000000    17.000000  27920.050000


In [12]:
# RFM Scores
rfm['R_score'] = pd.qcut(rfm['recency'], 5, labels=[5,4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])
rfm['M_score'] = pd.qcut(rfm['monetary'], 5, labels=[1,2,3,4,5])

# Support tickets
ticket_count = tickets.groupby('customer_id').size().reset_index(name='support_ticket_count')

# Merge support data
rfm = rfm.merge(ticket_count, on='customer_id', how='left')
rfm.fillna(0, inplace=True)

# Segments
def assign_segment(row):

    if int(row['R_score']) >= 4 and int(row['F_score']) >= 4 and int(row['M_score']) >= 4:
        return "Champions"

    elif int(row['F_score']) >= 4:
        return "Loyal Customers"

    elif int(row['R_score']) <= 2 and int(row['M_score']) >= 4:
        return "At Risk"

    elif int(row['R_score']) <= 2 and int(row['F_score']) <= 2:
        return "Dormant"

    elif row['support_ticket_count'] >= 2 and int(row['M_score']) >= 4:
        return "High Value Unhappy"

    else:
        return "Potential Loyalists"

rfm['segment_name'] = rfm.apply(assign_segment, axis=1)

print(rfm['segment_name'].value_counts())

segment_name
Potential Loyalists    847
Loyal Customers        520
Dormant                519
Champions              440
At Risk                 67
High Value Unhappy       7
Name: count, dtype: int64


In [13]:
rfm.sort_values(
    ['monetary','recency'],
    ascending=[False, False]
).head(10)

,customer_id,recency,frequency,monetary,R_score,F_score,M_score,support_ticket_count,segment_name
210,CUST00211,28,7,27920.05,4,5,5,1.0,Champions
1867,CUST01868,141,3,26057.29,2,3,5,0.0,At Risk
2105,CUST02106,32,9,22131.36,4,5,5,3.0,Champions
271,CUST00272,54,14,14120.16,3,5,5,0.0,Loyal Customers
1987,CUST01988,3,4,14032.75,5,3,5,0.0,Potential Loyalists
1294,CUST01295,182,4,12408.61,2,3,5,1.0,At Risk
2153,CUST02154,50,17,12407.64,3,5,5,3.0,Loyal Customers
1359,CUST01360,221,8,12310.69,1,5,5,2.0,Loyal Customers
816,CUST00817,43,15,12291.02,3,5,5,1.0,Loyal Customers
990,CUST00991,49,12,11918.47,3,5,5,2.0,Loyal Customers


In [14]:
import pandas as pd

customers = pd.read_csv("../source/customers.csv")
orders = pd.read_csv("../source/orders.csv")
tickets = pd.read_csv("../source/support_tickets.csv")
web = pd.read_csv("../source/web_events_snapshot.csv")
campaign = pd.read_csv("../source/intervention_history.csv")

In [15]:
import os
print(os.listdir("../source"))

['churn_labels.csv', 'customers.csv', 'DATA_DICTIONARY.md', 'intervention_history.csv', 'orders.csv', 'rfm_modeling_snapshot.csv', 'STUDENT_FACING_PROBLEM_STATEMENT.md', 'support_tickets.csv', 'web_events_snapshot.csv']


In [16]:
snapshot = pd.read_csv("../source/rfm_modeling_snapshot.csv")
churn = pd.read_csv("../source/churn_labels.csv")

print(snapshot.shape)
print(churn.shape)

print(snapshot.columns.tolist())
print(churn.columns.tolist())

(2400, 29)
(2400, 4)
['customer_id', 'snapshot_date', 'city_tier', 'age_group', 'acquisition_channel', 'loyalty_tier', 'preferred_category', 'marketing_consent', 'recency_days', 'frequency_180d', 'monetary_180d', 'return_rate_180d', 'avg_discount_pct_180d', 'avg_rating_180d', 'category_diversity_180d', 'ticket_count_90d', 'negative_ticket_rate_90d', 'avg_resolution_hours_90d', 'days_since_signup', 'sessions_30d', 'product_views_30d', 'cart_adds_30d', 'wishlist_adds_30d', 'abandoned_carts_30d', 'email_opens_30d', 'campaign_clicks_30d', 'last_visit_days_ago', 'churn_next_60d', 'split']
['customer_id', 'snapshot_date', 'churn_next_60d', 'split']


In [17]:
snapshot['churn_next_60d'].value_counts()

churn_next_60d
0    1273
1    1127
Name: count, dtype: int64

In [18]:
snapshot['churn_next_60d'].value_counts(normalize=True)*100

churn_next_60d
0    53.041667
1    46.958333
Name: proportion, dtype: float64

In [19]:
snapshot['split'].value_counts()

split
train         1728
validation     336
test           336
Name: count, dtype: int64

In [20]:
from sklearn.model_selection import train_test_split

df = snapshot.copy()

target = 'churn_next_60d'

drop_cols = [
    'customer_id',
    'snapshot_date',
    'split',
    'churn_next_60d'
]

X = df.drop(columns=drop_cols)
y = df[target]

X.head()

,city_tier,age_group,acquisition_channel,loyalty_tier,preferred_category,marketing_consent,recency_days,frequency_180d,monetary_180d,return_rate_180d,...,avg_resolution_hours_90d,days_since_signup,sessions_30d,product_views_30d,cart_adds_30d,wishlist_adds_30d,abandoned_carts_30d,email_opens_30d,campaign_clicks_30d,last_visit_days_ago
0,Tier 1,18-24,Instagram,Silver,Makeup,Yes,107,1,362.73,0.0,...,0.0,524,1,4,0,0,0,2,0,20
1,Tier 2,25-34,Marketplace,Silver,Hair Care,Yes,40,1,581.00,0.0,...,1.0,121,8,31,4,2,3,0,0,0
2,Tier 1,25-34,Influencer,NaN,Skin Care,Yes,171,1,649.98,0.0,...,0.0,206,1,3,0,0,0,0,0,26
3,Tier 3,25-34,Google Search,NaN,Fragrance,No,131,1,1604.04,0.0,...,0.0,168,1,6,0,0,0,0,0,14
4,Tier 3,35-44,Organic,Gold,Hair Care,Yes,38,3,1781.90,0.0,...,0.0,405,18,95,4,1,1,3,1,9


In [21]:
X = pd.get_dummies(X, drop_first=True)

print(X.shape)

(2400, 37)


In [22]:
train_df = snapshot[snapshot['split'] == 'train']
valid_df = snapshot[snapshot['split'] == 'validation']
test_df  = snapshot[snapshot['split'] == 'test']

train_idx = train_df.index
valid_idx = valid_df.index
test_idx  = test_df.index

X_train = X.loc[train_idx]
X_valid = X.loc[valid_idx]
X_test  = X.loc[test_idx]

y_train = y.loc[train_idx]
y_valid = y.loc[valid_idx]
y_test  = y.loc[test_idx]

print(X_train.shape)
print(X_valid.shape)
print(X_test.shape)

(1728, 37)
(336, 37)
(336, 37)


In [23]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=2000)

lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [24]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    random_state=42
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:,1]

In [25]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("Accuracy:", accuracy_score(y_test, rf_pred))
print("Precision:", precision_score(y_test, rf_pred))
print("Recall:", recall_score(y_test, rf_pred))
print("F1:", f1_score(y_test, rf_pred))
print("ROC AUC:", roc_auc_score(y_test, rf_prob))

Accuracy: 0.8065476190476191
Precision: 0.8410596026490066
Recall: 0.7559523809523809
F1: 0.7962382445141066
ROC AUC: 0.8840348639455782


In [26]:
importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf.feature_importances_
})

importance.sort_values(
    'importance',
    ascending=False
).head(15)

,feature,importance
0,recency_days,0.298289
18,last_visit_days_ago,0.151711
2,monetary_180d,0.093653
12,product_views_30d,0.045561
10,days_since_signup,0.044973
1,frequency_180d,0.043242
4,avg_discount_pct_180d,0.041587
6,category_diversity_180d,0.039770
11,sessions_30d,0.035719
5,avg_rating_180d,0.022003


In [27]:
snapshot['churn_next_60d'].value_counts()

churn_next_60d
0    1273
1    1127
Name: count, dtype: int64

In [28]:
snapshot['split'].value_counts()

split
train         1728
validation     336
test           336
Name: count, dtype: int64

In [29]:
print("Accuracy:", ...)
print("Precision:", ...)
print("Recall:", ...)
print("F1:", ...)
print("ROC AUC:", ...)

Accuracy: Ellipsis
Precision: Ellipsis
Recall: Ellipsis
F1: Ellipsis
ROC AUC: Ellipsis


In [30]:
importance.sort_values(
    'importance',
    ascending=False
).head(10)

,feature,importance
0,recency_days,0.298289
18,last_visit_days_ago,0.151711
2,monetary_180d,0.093653
12,product_views_30d,0.045561
10,days_since_signup,0.044973
1,frequency_180d,0.043242
4,avg_discount_pct_180d,0.041587
6,category_diversity_180d,0.039770
11,sessions_30d,0.035719
5,avg_rating_180d,0.022003


In [31]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("Accuracy:", accuracy_score(y_test, rf_pred))
print("Precision:", precision_score(y_test, rf_pred))
print("Recall:", recall_score(y_test, rf_pred))
print("F1:", f1_score(y_test, rf_pred))
print("ROC AUC:", roc_auc_score(y_test, rf_prob))

Accuracy: 0.8065476190476191
Precision: 0.8410596026490066
Recall: 0.7559523809523809
F1: 0.7962382445141066
ROC AUC: 0.8840348639455782


In [32]:
import joblib

joblib.dump(rf, "model.pkl")

['model.pkl']

In [33]:
import os
print(os.path.exists("model.pkl"))

True


In [34]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, rf_pred)
print(cm)

[[144  24]
 [ 41 127]]


In [35]:
results = pd.DataFrame({
    "customer_id": snapshot.loc[test_idx, "customer_id"],
    "actual": y_test,
    "predicted": rf_pred,
    "probability": rf_prob
})

false_positive = results[
    (results.actual == 0) &
    (results.predicted == 1)
]

false_negative = results[
    (results.actual == 1) &
    (results.predicted == 0)
]

print(false_positive.head())
print(false_negative.head())

    customer_id  actual  predicted  probability
108   CUST00109       0          1     0.596871
334   CUST00335       0          1     0.698835
436   CUST00437       0          1     0.892565
490   CUST00491       0          1     0.638708
814   CUST00815       0          1     0.733395
    customer_id  actual  predicted  probability
66    CUST00067       1          0     0.429444
87    CUST00088       1          0     0.465259
183   CUST00184       1          0     0.073990
246   CUST00247       1          0     0.354857
378   CUST00379       1          0     0.404799


In [36]:
# Final Segment Summary

segment_summary = rfm.groupby('segment_name').agg({
    'customer_id': 'count',
    'recency': 'mean',
    'frequency': 'mean',
    'monetary': 'mean',
    'support_ticket_count': 'mean'
}).round(2)

segment_summary.columns = [
    'Customer_Count',
    'Avg_Recency',
    'Avg_Frequency',
    'Avg_Monetary',
    'Avg_Support_Tickets'
]

segment_summary = segment_summary.sort_values(
    'Customer_Count',
    ascending=False
)

segment_summary

,Customer_Count,Avg_Recency,Avg_Frequency,Avg_Monetary,Avg_Support_Tickets
segment_name,,,,,
Potential Loyalists,847,67.77,2.83,2012.75,0.46
Loyal Customers,520,117.56,6.32,4502.69,1.29
Dormant,519,217.66,1.60,1123.55,0.40
Champions,440,19.90,7.35,5679.80,1.31
At Risk,67,226.43,3.55,4312.69,0.88
High Value Unhappy,7,42.57,3.71,4042.04,2.29


In [37]:
rfm.to_csv("segments.csv", index=False)